<a href="https://colab.research.google.com/github/AnDinh039/PST-PRIM---RMAT_128.edges---Julia/blob/main/L%E1%BB%97i_Datalog_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install "pycozo[embedded,requests,pandas]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 95.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 97.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [cozo-embedded]


In [28]:
from pycozo.client import Client

client = Client()

# Tự tạo dataset đơn giản: 10 dòng xtrain(id, c, v) (5 mẫu x 2 đặc trưng c=0, 1)
# Hàm mục tiêu: y = 2*x0 + 3*x1 | Khởi tạo model0(c, p): (0, 22.0) và (1, 43.0) giống Hình 1b
client.run("""
xtrain[id, c, v] <- [
    [1, 0, 1.0], [1, 1, 2.0],
    [2, 0, 2.0], [2, 1, 1.0],
    [3, 0, 3.0], [3, 1, 4.0],
    [4, 0, 4.0], [4, 1, 3.0],
    [5, 0, 5.0], [5, 1, 5.0]
]
ytrain[id, y] <- [[1, 8.0], [2, 7.0], [3, 18.0], [4, 17.0], [5, 25.0]]
model0[c, p]  <- [[0, 22.0], [1, 43.0]]

# Rule 1: model(t, c, p) :- (t = 0), model0(c, p).
model[t, c, p] := t = 0, model0[c, p]

# Rule 2: predict(t, id, sum(y)) :- model(t, c, p), xtrain(id, c, v), y = v * p.
predict[t, id, sum(y)] := model[t, c, p], xtrain[id, c, v], y = v * p

# Rule 3: gradient(t, c, sum(g)) :- ytrain(id, y), predict(t, id, y_pred), xtrain(id, c, v), g = 2*(y_pred - y)*v.
gradient[t, c, sum(g)] := ytrain[id, y], predict[t, id, y_pred], xtrain[id, c, v], g = 2.0 * (y_pred - y) * v

# Rule 4: model(t+1, c, p') :- model(t, c, p), gradient(t, c, g), (t < 5), p' = p - (0.01 * g / 5).
model[t_next, c, p_next] := model[t, c, p], gradient[t, c, g], t < 5, t_next = t + 1, p_next = p - (0.01 * g / 5.0)

?[t, c, p] := model[t, c, p]
""")





QueryException: Query is unstratifiable

In [29]:

from pycozo.client import Client

client = Client()

client.run("""
xtrain[id, c, v] <- [
    [1, 0, 1.0], [1, 1, 2.0],
    [2, 0, 2.0], [2, 1, 1.0],
    [3, 0, 3.0], [3, 1, 4.0],
    [4, 0, 4.0], [4, 1, 3.0],
    [5, 0, 5.0], [5, 1, 5.0]
]
ytrain[id, y] <- [[1, 8.0], [2, 7.0], [3, 18.0], [4, 17.0], [5, 25.0]]
model0[c, p]  <- [[0, 22.0], [1, 43.0]]

model[t, c, p] := t = 0, model0[c, p]
predict[t, id, sum(y)] := model[t, c, p], xtrain[id, c, v], y = v * p
gradient[t, c, sum(g)] := ytrain[id, y], predict[t, id, y_pred], xtrain[id, c, v], g = 2.0 * (y_pred - y) * v

# Sai quy tắc thời gian: cập nhật ngược vào cùng thời điểm t thay vì t + 1
model[t, c, p_next] := model[t, c, p], gradient[t, c, g], p_next = p - (0.01 * g / 5.0)

?[t, c, p] := model[t, c, p]
""")


QueryException: Query is unstratifiable

In [31]:
import time
from pycozo.client import Client

# Cấu hình bài toán BGD
MAX_ITER = 5          # Chạy 5 vòng lặp (t = 0..5) để in chi tiết từng bước như Hình 1b
ALPHA = 0.02          # Learning rate
N_SAMPLES = 5.0       # Số mẫu huấn luyện (5 mẫu x 2 đặc trưng = 10 dòng xtrain)

XTRAIN_DATA = [
    [1, 0, 1.0], [1, 1, 2.0],
    [2, 0, 2.0], [2, 1, 1.0],
    [3, 0, 3.0], [3, 1, 4.0],
    [4, 0, 4.0], [4, 1, 3.0],
    [5, 0, 5.0], [5, 1, 5.0],
]
YTRAIN_DATA = [[1, 8.0], [2, 7.0], [3, 18.0], [4, 17.0], [5, 25.0]]
MODEL0_DATA = [[0, 22.0], [1, 43.0]]  # Giống hệt Hình 1b (trang 3): (0, 22) và (1, 43)


# ==============================================================================
# 1. FREEDATALOG TRÊN COZODB (Hình 1b - Bảng trên: Naïve Evaluation)
# ==============================================================================
def run_freedatalog_cozo(max_iter=MAX_ITER, verbose=True):
    # Tắt chế độ tự chuyển sang DataFrame để lấy trực tiếp dict["rows"]
    client = Client(dataframe=False)

    # Khởi tạo bảng EDB (10 dòng xtrain, 5 dòng ytrain, 2 dòng model0) và các bảng IDB phẳng
    client.run("?[id, c, v] <- $d :create xtrain {id: Int, c: Int => v: Float}", {"d": XTRAIN_DATA})
    client.run("?[id, y] <- $d :create ytrain {id: Int => y: Float}", {"d": YTRAIN_DATA})
    client.run("?[c, p] <- $d :create model0 {c: Int => p: Float}", {"d": MODEL0_DATA})

    client.run(":create model {t: Int, c: Int => p: Float}")
    client.run(":create predict {t: Int, id: Int => y_pred: Float}")
    client.run(":create gradient {t: Int, c: Int => g: Float}")

    t_start = time.perf_counter()

    # Bước cơ sở (Iter #0): model(t, c, p) :- (t = 0), model0(c, p).
    client.run("""
        ?[t, c, p] := t = 0, *model0[c, p]
        :put model {t, c => p}
    """)

    if verbose:
        print("=" * 95)
        print("[B1] DIỄN GIẢI TỪNG BƯỚC: FREEDATALOG (HÌNH 1B - BẢNG TRÊN) CHẠY TRÊN COZODB")
        print("=" * 95)
        print("  [Iter #0] Khởi tạo model(0, c, p) từ model0: c=0 -> p=22.00, c=1 -> p=43.00")

    total_recomputed_tuples = 0

    for it in range(max_iter):
        # 1. Tính predict(t, id, sum(y)) trên TOÀN BỘ bảng model(t, c, p) hiện có (mọi t = 0..it)
        pred_res = client.run("""
            ?[t, id, sum(y)] := *model[t, c, p], *xtrain[id, c, v], y = v * p
        """)
        client.run("?[t, id, y_pred] <- $d :put predict {t, id => y_pred}", {"d": pred_res["rows"]})

        # 2. Tính gradient(t, c, sum(g)) trên TOÀN BỘ predict(t, id, y_pred) (mọi t = 0..it)
        grad_res = client.run("""
            ?[t, c, sum(g)] := *ytrain[id, y], *predict[t, id, y_pred], *xtrain[id, c, v],
                               g = 2.0 * (y_pred - y) * v
        """)
        client.run("?[t, c, g] <- $d :put gradient {t, c => g}", {"d": grad_res["rows"]})

        # 3. Suy diễn model(t+1, c, p') cho MỌI t = 0..it đang có trong bảng
        next_model_res = client.run(f"""
            ?[t_next, c, p_next] := *model[t, c, p], *gradient[t, c, g],
                                    t < {max_iter}, t_next = t + 1,
                                    p_next = p - ({ALPHA} * g / {N_SAMPLES})
        """)
        client.run("?[t, c, p] <- $d :put model {t, c => p}", {"d": next_model_res["rows"]})

        recomputed_step = len(pred_res["rows"]) + len(grad_res["rows"]) + len(next_model_res["rows"])
        total_recomputed_tuples += recomputed_step

        if verbose:
            all_models = client.run("?[t, c, p] := *model[t, c, p] :order t, c")["rows"]
            latest_w = [r for r in all_models if r[0] == it + 1]
            recomputed_ts = sorted(list({r[0] for r in grad_res["rows"]}))
            print(
                f"  [Iter #{it+1}] CozoDB phải tính lại cho mọi t={recomputed_ts} ({recomputed_step:2d} tuples tính lại) | "
                f"Sinh ra model(t={it+1}): c=0 -> p={latest_w[0][2]:7.2f}, c=1 -> p={latest_w[1][2]:7.2f}"
            )

    exec_time_ms = (time.perf_counter() - t_start) * 1000.0

    # Đếm tổng số dòng (tuples) đang lưu trữ trong các bảng IDB của CozoDB
    n_model = len(client.run("?[t, c, p] := *model[t, c, p]")["rows"])
    n_pred  = len(client.run("?[t, id, y] := *predict[t, id, y]")["rows"])
    n_grad  = len(client.run("?[t, c, g] := *gradient[t, c, g]")["rows"])
    total_stored_tuples = n_model + n_pred + n_grad
    ram_bytes = total_stored_tuples * 24

    final_model = client.run(f"?[c, p] := *model[t, c, p], t == {max_iter} :order c")["rows"]
    return final_model, total_stored_tuples, total_recomputed_tuples, ram_bytes, exec_time_ms


# ==============================================================================
# 2. TEMPODL TRÊN COZODB (Hình 1b - Bảng dưới & Hình 7b: Tối ưu hóa trên TempoDL)
# ==============================================================================
def run_tempodl_cozo(max_iter=MAX_ITER, verbose=True):
    client = Client(dataframe=False)

    client.run("?[id, c, v] <- $d :create xtrain {id: Int, c: Int => v: Float}", {"d": XTRAIN_DATA})
    client.run("?[id, y] <- $d :create ytrain {id: Int => y: Float}", {"d": YTRAIN_DATA})
    client.run("?[c, p] <- $d :create model0 {c: Int => p: Float}", {"d": MODEL0_DATA})

    # Trong TempoDL đã tối ưu: Bảng IDB không còn cột `t` và chỉ lưu trạng thái hiện tại
    client.run(":create model {c: Int => p: Float}")
    client.run(":create predict {id: Int => y_pred: Float}")
    client.run(":create gradient {c: Int => g: Float}")

    t_start = time.perf_counter()

    # Bước cơ sở (t = 0): model(c, p) := model0(c, p)
    client.run("""
        ?[c, p] := *model0[c, p]
        :put model {c => p}
    """)

    if verbose:
        print("\n" + "=" * 95)
        print("[B2] DIỄN GIẢI TỪNG BƯỚC: TEMPODL (HÌNH 1B - BẢNG DƯỚI & HÌNH 7B) CHẠY TRÊN COZODB")
        print("=" * 95)
        print("  [Bước t=0] Khởi tạo model(c, p) từ model0: c=0 -> p=22.00, c=1 -> p=43.00")

    total_computed_tuples = 0

    # Vòng lặp TempoDL: for t from 0 ... end
    for t in range(max_iter):
        # 1. Chỉ tính predict(id, sum(y)) cho bước hiện tại
        pred_res = client.run("""
            ?[id, sum(y)] := *model[c, p], *xtrain[id, c, v], y = v * p
        """)
        client.run("?[id, y_pred] <- $d :put predict {id => y_pred}", {"d": pred_res["rows"]})

        # 2. Chỉ tính gradient(c, sum(g)) cho bước hiện tại
        grad_res = client.run("""
            ?[c, sum(g)] := *ytrain[id, y], *predict[id, y_pred], *xtrain[id, c, v],
                            g = 2.0 * (y_pred - y) * v
        """)
        client.run("?[c, g] <- $d :put gradient {c => g}", {"d": grad_res["rows"]})

        # 3. Cập nhật trực tiếp model(c, p') cho bước kế tiếp
        next_model_res = client.run(f"""
            ?[c, p_next] := *model[c, p], *gradient[c, g],
                            p_next = p - ({ALPHA} * g / {N_SAMPLES})
        """)
        client.run("?[c, p] <- $d :put model {c => p}", {"d": next_model_res["rows"]})

        step_tuples = len(pred_res["rows"]) + len(grad_res["rows"]) + len(next_model_res["rows"])
        total_computed_tuples += step_tuples

        if verbose:
            curr_w = sorted(next_model_res["rows"], key=lambda r: r[0])
            print(
                f"  [Bước t={t+1}] Chỉ tính đúng tầng hiện tại ({step_tuples:2d} tuples)           | "
                f"Cập nhật model:      c=0 -> p={curr_w[0][1]:7.2f}, c=1 -> p={curr_w[1][1]:7.2f}"
            )

    exec_time_ms = (time.perf_counter() - t_start) * 1000.0

    # Đếm tổng số dòng (tuples) đang lưu trữ trong CozoDB
    n_model = len(client.run("?[c, p] := *model[c, p]")["rows"])
    n_pred  = len(client.run("?[id, y] := *predict[id, y]")["rows"])
    n_grad  = len(client.run("?[c, g] := *gradient[c, g]")["rows"])
    total_stored_tuples = n_model + n_pred + n_grad
    ram_bytes = total_stored_tuples * 16

    final_model = client.run("?[c, p] := *model[c, p] :order c")["rows"]
    return final_model, total_stored_tuples, total_computed_tuples, ram_bytes, exec_time_ms


# ==============================================================================
# CHẠY MINH HỌA 5 BƯỚC & ĐO LƯỜNG TRÊN 50 VÒNG LẶP HUẤN LUYỆN
# ==============================================================================
run_freedatalog_cozo(max_iter=5, verbose=True)
run_tempodl_cozo(max_iter=5, verbose=True)

# Đo hiệu năng trên 50 vòng lặp BGD (MAX_ITER = 50)
m1, stored1, comp1, ram1, time1 = run_freedatalog_cozo(max_iter=50, verbose=False)
m2, stored2, comp2, ram2, time2 = run_tempodl_cozo(max_iter=50, verbose=False)

print("\n" + "=" * 95)
print("BẢNG TỔNG KẾT TRÊN COZODB (BÀI TOÁN BGD - 50 VÒNG LẶP HUẤN LUYỆN, DATASET 10 DÒNG XTRAIN)")
print("=" * 95)
print(f"{'Phương pháp trên CozoDB':<30} | {'Tuples tính toán':>16} | {'Tuples lưu RAM':>14} | {'Bộ nhớ IDB':>12} | {'Thời gian':>10}")
print("-" * 95)
print(f"{'1. FreeDatalog (Hình 1b trên)':<30} | {comp1:>16,d} | {stored1:>14,d} | {ram1:>10,d} B | {time1:>7.2f} ms")
print(f"{'2. TempoDL (Hình 1b dưới)':<30} | {comp2:>16,d} | {stored2:>14,d} | {ram2:>10,d} B | {time2:>7.2f} ms")
print("=" * 95)
print(f"Trọng số hội tụ cuối cùng (giống nhau cả 2 bản): c=0 -> p={m2[0][1]:.4f}, c=1 -> p={m2[1][1]:.4f}")





[B1] DIỄN GIẢI TỪNG BƯỚC: FREEDATALOG (HÌNH 1B - BẢNG TRÊN) CHẠY TRÊN COZODB
  [Iter #0] Khởi tạo model(0, c, p) từ model0: c=0 -> p=22.00, c=1 -> p=43.00
  [Iter #1] CozoDB phải tính lại cho mọi t=[0] ( 9 tuples tính lại) | Sinh ra model(t=1): c=0 -> p=  -3.76, c=1 -> p=  16.92
  [Iter #2] CozoDB phải tính lại cho mọi t=[0, 1] (18 tuples tính lại) | Sinh ra model(t=2): c=0 -> p=  -7.13, c=1 -> p=  13.24
  [Iter #3] CozoDB phải tính lại cho mọi t=[0, 1, 2] (27 tuples tính lại) | Sinh ra model(t=3): c=0 -> p=  -7.45, c=1 -> p=  12.60
  [Iter #4] CozoDB phải tính lại cho mọi t=[0, 1, 2, 3] (36 tuples tính lại) | Sinh ra model(t=4): c=0 -> p=  -7.36, c=1 -> p=  12.39
  [Iter #5] CozoDB phải tính lại cho mọi t=[0, 1, 2, 3, 4] (45 tuples tính lại) | Sinh ra model(t=5): c=0 -> p=  -7.22, c=1 -> p=  12.23

[B2] DIỄN GIẢI TỪNG BƯỚC: TEMPODL (HÌNH 1B - BẢNG DƯỚI & HÌNH 7B) CHẠY TRÊN COZODB
  [Bước t=0] Khởi tạo model(c, p) từ model0: c=0 -> p=22.00, c=1 -> p=43.00
  [Bước t=1] Chỉ tính đúng tần